# State Estimation with Physics-Encoded Neural Network 

Using a Context-Aware DeLaN model with log-Cholesky decomposition for inertia matrix and torque biases estimation, and residual torque prediction. Use inertia matrix and torque biases in Kalman Filter for state estimation.

Currently working with simulated data by `mpx`.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

seeds = range(5)

dataset_num = 8
dataset_path = Path.cwd().parent / f"custom_datasets/quad_mass_dataset_run{dataset_num}.npz"

### Evalaute Input Data / Model

The input data for the LSTM could be either [joint_pos, joint_vel, tau_diff] or [base_orient, base_lin_vel, base_ang_vel, base_pos_z, tau_diff]. 

Evaluate performance of both options across all seeds. Log all data to use in a table in the report, determine best model.

In [ ]:
from utils.evaluate import evaluate_dataset, print_report

for input_values in ["joint", "base_pos_z"]:
    for seed in seeds:
        model_name = f"epochs_1000_quad_mass_dataset_run8_input_values_{input_values}_16-16_lstm5x10_wd1e-05_span0.5s-stride4-gap10_seed_{seed}"
        cadelac_path = Path(f"trained_models/LogChol-CaDeLaC/LogChol-CaDeLaC/{model_name}")

        res = evaluate_dataset(dataset_path, cadelac_path, verbose=False)
        print_report(res)

### Evalaute KF estimation

The state estimation is tested with these methods:

1. Leg odometry alone
2. KF with nominal model (use only torque, inertia matrix, qfrc_bias for without payload)
3. KF with LogChol-CaDeLaC (proposed approach)
4. KF with gt residual model (ground truth values from simulation)

For each state parameter, plot the estimation for each method, along with the ground truth.

For each method, plot the estimation for each state parameter 

In [ ]:
from utils.kf_utils import load_custom_dataset
from utils.report_plots import run_ladder_for_sim

input_values = None
model_name = f"epochs_1000_quad_mass_dataset_run8_input_values_{input_values}_16-16_lstm5x10_wd1e-05_span0.5s-stride4-gap10_seed_{seed}"
cadelac_path = Path(f"trained_models/LogChol-CaDeLaC/LogChol-CaDeLaC/{model_name}")

sim_num = 2
data = load_custom_dataset(dataset_path=dataset_path, sim_num=sim_num)

ladder = run_ladder_for_sim(data, cadelac_path)

In [ ]:
from utils.report_plots import plot_state_comparison

fig_dir = Path("figures/report") / model_name
figs = plot_state_comparison(data, ladder, sim_label=f"sim-{sim_num}", save_dir=fig_dir)